    **Dữ liệu:**

1. movies.txt
- **Schema**: MovieID, Title, Genres

2. ratings_1.txt, ratings_2.txt
- **Schema**: UserID, MovieID, Rating, Timestamp

3. users.txt
- **Schema**: UserID, Gender, Age, Occupation, Zip-code

4. occupation.txt
- **Schema**: ID, Occupation

**Lưu ý:** Thực hiện các bài tập dưới đây sử dụng RDD.

**Bài 1**: Tính điểm trung bình và tổng số lượt đánh giá cho mỗi phim
* **Mục tiêu**:
  * Tính điểm trung bình của mỗi phim.
  * Đếm tổng số lượt đánh giá.
  * Tìm phim có điểm trung bình cao nhất (chỉ xét những phim có ít nhất 50 lượt đánh giá).
* **Giải pháp**:
  * Bước 1: Đọc file movies.txt và tạo một map (MovieID → Title).
  * Bước 2: Đọc file ratings_1.txt và ratings_2.txt, map MovieID → (Rating, 1).
  * Bước 3: Reduce để tính tổng điểm và số lượt đánh giá.
  * Bước 4: Tính điểm trung bình, lọc ra phim có ít nhất 5 lượt đánh giá.
  * Bước 5: Tìm phim có điểm trung bình cao nhất.

Bài 2: Phân tích đánh giá theo thể loại
* Mục tiêu:
  * Tính điểm trung bình của từng thể loại phim.
* Giải pháp
  * Bước 1: Tạo map (MovieID → List of Genres).
  * Bước 2: Map từ MovieID → Rating → (Genre, Rating).
  * Bước 3: Tính trung bình điểm đánh giá cho từng thể loại.

Bài 3: Phân tích đánh giá theo giới tính
* Mục tiêu:
  * Tính điểm trung bình của mỗi phim theo giới tính.
* Giải pháp
  * Bước 1: Tạo map (UserID → Gender).
  * Bước 2: Join với ratings để thêm thông tin giới tính.
  * Bước 3: Tính trung bình rating cho mỗi phim theo từng giới tính.

Bài 4: Phân tích đánh giá theo nhóm tuổi
* Mục tiêu:
  * Phân loại người dùng theo nhóm tuổi và tính điểm trung bình của mỗi phim theo từng nhóm.
* Giải pháp
  * Bước 1: Tạo map (UserID → Age Group).
  * Bước 2: Join với ratings để thêm nhóm tuổi.
  * Bước 3: Tính trung bình điểm đánh giá theo nhóm tuổi.

Bài 5: Phân Tích Đánh Giá Theo Occupation (Nghề nghiệp) Của Người Dùng
* Mục tiêu:
  * Tính trung bình rating và tổng số lượt đánh giá cho từng Occupation.
* Giải pháp:
  * Tạo dictionary từ users.txt với mapping UserID → Occupation.
  * Với mỗi rating, gán thông tin Occupation theo UserID.
  * Phát hành cặp key-value với key là Occupation và value là (rating, 1).
  * Reduce để tính tổng điểm và số lượt cho mỗi Occupation, sau đó tính trung bình rating.

Bài 6: Phân Tích Đánh Giá Theo Thời Gian
* Mục tiêu:
  * Tính tổng số lượt đánh giá và điểm trung bình cho mỗi năm.
* Giải pháp:
  * Đọc dữ liệu ratings (từ cả ratings_1.txt và ratings_2.txt).
  * Sử dụng hàm trợ giúp để chuyển đổi Timestamp (dạng Unix) thành năm (Year).
  * Với mỗi dòng ratings, phát hành cặp key-value với key là năm và value là (rating, 1).
  * Reduce để tính tổng điểm và số lượt cho mỗi năm, sau đó tính trung bình.

# Set up

# __Assignment 1__

In [1]:
!mkdir -p assignment1
!mkdir -p assignment2
!mkdir -p assignment3
!mkdir -p assignment4
!mkdir -p assignment5
!mkdir -p assignment6

In [2]:
%%bash
cat <<EOF > assignment1/Assignment1.py

from pyspark import SparkContext

sc = SparkContext(appName="Assignment1")
sc.setLogLevel("ERROR")

def print_table(data, headers):
    data_str = [[str(x) for x in row] for row in data]
    cols = list(zip(*([headers] + data_str)))
    col_widths = [max(len(item) for item in col) for col in cols]
    def format_row(row):
        return " | ".join(val.ljust(width) for val, width in zip(row, col_widths))
    print(format_row(headers))
    print("-+-".join('-'*w for w in col_widths))
    for row in data_str:
        print(format_row(row))

movies = (
  sc.textFile("movies.txt")
  .map(lambda x: x.split(","))
  .map(lambda x: (x[0], x[1]))
)

ratings = (sc.textFile("ratings_1.txt")
    .union(sc.textFile("ratings_2.txt"))
    .map(lambda x: x.split(","))
    .map(lambda x: (x[1], (float(x[2]), 1)))
)

agg = ratings.reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))
avg = agg.mapValues(lambda x: (x[0]/x[1], x[1]))
filtered = avg.filter(lambda x: x[1][1] >= 5)

result = (
  filtered.join(movies)
    .map(lambda x: (x[0], x[1][1], round(x[1][0][0],2), x[1][0][1]))
)

# 👉 collect để in bảng
top = result.take(20)
print_table(top, ["MovieID","Title","AvgRating","Count"])

# save
output = result.map(lambda x: f"{x[0]},{x[1]},{x[2]},{x[3]}")
output.coalesce(1).saveAsTextFile("assignment1/output")

EOF

In [3]:
!rm -rf assignment1/output
!spark-submit assignment1/Assignment1.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 11:50:55 INFO SparkContext: Running Spark version 4.0.2
26/04/22 11:50:55 INFO SparkContext: OS info Linux, 6.6.113+, amd64
26/04/22 11:50:55 INFO SparkContext: Java version 17.0.18
26/04/22 11:50:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/22 11:50:55 INFO ResourceUtils: ==============================================================
26/04/22 11:50:55 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/22 11:50:55 INFO ResourceUtils: ==============================================================
26/04/22 11:50:55 INFO SparkContext: Submitted application: Assignment1
26/04/22 11:50:55 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name:

# __Assignment 2__

In [4]:
%%bash
cat <<EOF > assignment2/Assignment2.py

from pyspark import SparkContext

sc = SparkContext(appName="Assignment2")

def print_table(data, headers):
    data_str = [[str(x) for x in row] for row in data]
    cols = list(zip(*([headers] + data_str)))
    col_widths = [max(len(item) for item in col) for col in cols]
    def format_row(row):
        return " | ".join(val.ljust(width) for val, width in zip(row, col_widths))
    print(format_row(headers))
    print("-+-".join('-'*w for w in col_widths))
    for row in data_str:
        print(format_row(row))

movies = sc.textFile("movies.txt") \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], x[2].split("|")))

ratings = sc.textFile("ratings_1.txt") \
    .union(sc.textFile("ratings_2.txt")) \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[1], float(x[2])))

joined = ratings.join(movies)

genre_ratings = joined.flatMap(
    lambda x: [(g, (x[1][0],1)) for g in x[1][1]]
)

agg = genre_ratings.reduceByKey(lambda a,b:(a[0]+b[0], a[1]+b[1]))
result = agg.mapValues(lambda x: round(x[0]/x[1],2)).sortBy(lambda x: x[0])

top = result.take(20)
print_table(top, ["Genre","AvgRating"])

output = result.map(lambda x: f"{x[0]},{x[1]}")
output.coalesce(1).saveAsTextFile("assignment2/output")
EOF

In [5]:
!rm -rf assignment2/output
!spark-submit assignment2/Assignment2.py movies.txt ratings_1.txt ratings_2.txt

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 11:51:16 INFO SparkContext: Running Spark version 4.0.2
26/04/22 11:51:16 INFO SparkContext: OS info Linux, 6.6.113+, amd64
26/04/22 11:51:16 INFO SparkContext: Java version 17.0.18
26/04/22 11:51:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/22 11:51:16 INFO ResourceUtils: ==============================================================
26/04/22 11:51:16 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/22 11:51:16 INFO ResourceUtils: ==============================================================
26/04/22 11:51:16 INFO SparkContext: Submitted application: Assignment2
26/04/22 11:51:16 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name:

# __Assignment 3__

In [6]:
%%bash
cat <<EOF > assignment3/Assignment3.py

from pyspark import SparkContext

sc = SparkContext(appName="Assignment3")

def print_table(data, headers):
    data_str = [[str(x) for x in row] for row in data]
    cols = list(zip(*([headers] + data_str)))
    col_widths = [max(len(item) for item in col) for col in cols]
    def format_row(row):
        return " | ".join(val.ljust(width) for val, width in zip(row, col_widths))
    print(format_row(headers))
    print("-+-".join('-'*w for w in col_widths))
    for row in data_str:
        print(format_row(row))

users = sc.textFile("users.txt") \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], x[1]))

movies = sc.textFile("movies.txt") \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], x[1]))

ratings = sc.textFile("ratings_1.txt") \
    .union(sc.textFile("ratings_2.txt")) \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], (x[1], float(x[2]))))

result = ratings.join(users) \
    .map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1],1))) \
    .reduceByKey(lambda a,b:(a[0]+b[0], a[1]+b[1])) \
    .mapValues(lambda x: (round(x[0]/x[1],2), x[1])) \
    .map(lambda x: (x[0][0], (x[0][1], x[1][0], x[1][1]))) \
    .join(movies) \
    .map(lambda x: (x[0], x[1][1], x[1][0][0], x[1][0][1], x[1][0][2])) \
    .sortBy(lambda x: (x[0], x[2]))

top = result.take(20)
print_table(top, ["MovieID","Title","Gender","AvgRating","Count"])

output = result.map(lambda x: f"{x[0]},{x[1]},{x[2]},{x[3]},{x[4]}")
output.coalesce(1).saveAsTextFile("assignment3/output")
EOF

In [7]:
!rm -rf assignment3/output
!spark-submit assignment3/Assignment3.py movies.txt ratings_1.txt ratings_2.txt users.txt

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 11:51:36 INFO SparkContext: Running Spark version 4.0.2
26/04/22 11:51:36 INFO SparkContext: OS info Linux, 6.6.113+, amd64
26/04/22 11:51:36 INFO SparkContext: Java version 17.0.18
26/04/22 11:51:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/22 11:51:36 INFO ResourceUtils: ==============================================================
26/04/22 11:51:36 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/22 11:51:36 INFO ResourceUtils: ==============================================================
26/04/22 11:51:36 INFO SparkContext: Submitted application: Assignment3
26/04/22 11:51:36 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name:

# __Assignment 4__

In [8]:
%%bash
cat <<EOF > assignment4/Assignment4.py

from pyspark import SparkContext

sc = SparkContext(appName="Assignment4")

def age_group(age):
    age = int(age)
    if age < 18: return "Under18"
    elif age < 35: return "18-34"
    elif age < 50: return "35-49"
    else: return "50+"

def print_table(data, headers):
    data_str = [[str(x) for x in row] for row in data]
    cols = list(zip(*([headers] + data_str)))
    col_widths = [max(len(item) for item in col) for col in cols]
    def format_row(row):
        return " | ".join(val.ljust(width) for val, width in zip(row, col_widths))
    print(format_row(headers))
    print("-+-".join('-'*w for w in col_widths))
    for row in data_str:
        print(format_row(row))

users = sc.textFile("users.txt") \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], age_group(x[2])))

movies = sc.textFile("movies.txt") \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], x[1]))   # movie_id -> title

ratings = sc.textFile("ratings_1.txt") \
    .union(sc.textFile("ratings_2.txt")) \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], (x[1], float(x[2]))))

result = ratings.join(users) \
    .map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1],1))) \
    .reduceByKey(lambda a,b:(a[0]+b[0], a[1]+b[1])) \
    .mapValues(lambda x: (round(x[0]/x[1],2), x[1])) \
    .map(lambda x: (x[0][0], (x[0][1], x[1][0], x[1][1]))) \
    .join(movies) \
    .map(lambda x: (x[0], x[1][1], x[1][0][0], x[1][0][1], x[1][0][2])) \
    .sortBy(lambda x: (x[0], x[2]))

top = result.take(20)
print_table(top, ["MovieID","Title","AgeGroup","AvgRating","Count"])

output = result.map(lambda x: f"{x[0]},{x[1]},{x[2]},{x[3]},{x[4]}")
output.coalesce(1).saveAsTextFile("assignment4/output")

EOF

In [9]:
!rm -rf assignment4/output
!spark-submit assignment4/Assignment4.py movies.txt ratings_1.txt ratings_2.txt users.txt

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 11:52:01 INFO SparkContext: Running Spark version 4.0.2
26/04/22 11:52:01 INFO SparkContext: OS info Linux, 6.6.113+, amd64
26/04/22 11:52:01 INFO SparkContext: Java version 17.0.18
26/04/22 11:52:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/22 11:52:01 INFO ResourceUtils: ==============================================================
26/04/22 11:52:01 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/22 11:52:01 INFO ResourceUtils: ==============================================================
26/04/22 11:52:01 INFO SparkContext: Submitted application: Assignment4
26/04/22 11:52:01 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name:

# __Assignment 5__

In [10]:
%%bash
cat <<EOF > assignment5/Assignment5.py

from pyspark import SparkContext

sc = SparkContext(appName="Assignment5")

def print_table(data, headers):
    data_str = [[str(x) for x in row] for row in data]
    cols = list(zip(*([headers] + data_str)))
    col_widths = [max(len(item) for item in col) for col in cols]
    def format_row(row):
        return " | ".join(val.ljust(width) for val, width in zip(row, col_widths))
    print(format_row(headers))
    print("-+-".join('-'*w for w in col_widths))
    for row in data_str:
        print(format_row(row))

users = sc.textFile("users.txt") \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], x[3]))

occupations = sc.textFile("occupation.txt") \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], x[1]))

ratings = sc.textFile("ratings_1.txt") \
    .union(sc.textFile("ratings_2.txt")) \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (x[0], float(x[2])))

result = ratings.join(users) \
    .map(lambda x: (x[1][1], (x[1][0], 1))) \
    .reduceByKey(lambda a,b:(a[0]+b[0], a[1]+b[1])) \
    .mapValues(lambda x: (round(x[0]/x[1],2), x[1])) \
    .join(occupations) \
    .map(lambda x: (x[1][1], x[1][0][0], x[1][0][1])) \
    .sortBy(lambda x: x[0])

top = result.take(20)
print_table(top, ["Occupation","AvgRating","Count"])

output = result.map(lambda x: f"{x[0]},{x[1]},{x[2]}")
output.coalesce(1).saveAsTextFile("assignment5/output")
EOF

In [11]:
!rm -rf assignment5/output
!spark-submit assignment5/Assignment5.py movies.txt ratings_1.txt ratings_2.txt users.txt occupation.txt

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 11:52:29 INFO SparkContext: Running Spark version 4.0.2
26/04/22 11:52:29 INFO SparkContext: OS info Linux, 6.6.113+, amd64
26/04/22 11:52:29 INFO SparkContext: Java version 17.0.18
26/04/22 11:52:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/22 11:52:29 INFO ResourceUtils: ==============================================================
26/04/22 11:52:29 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/22 11:52:29 INFO ResourceUtils: ==============================================================
26/04/22 11:52:29 INFO SparkContext: Submitted application: Assignment5
26/04/22 11:52:29 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name:

# __Assignment 6__

In [12]:
%%bash
cat <<EOF > assignment6/Assignment6.py

from pyspark import SparkContext
from datetime import datetime

sc = SparkContext(appName="Assignment6")

def print_table(data, headers):
    data_str = [[str(x) for x in row] for row in data]
    cols = list(zip(*([headers] + data_str)))
    col_widths = [max(len(item) for item in col) for col in cols]
    def format_row(row):
        return " | ".join(val.ljust(width) for val, width in zip(row, col_widths))
    print(format_row(headers))
    print("-+-".join('-'*w for w in col_widths))
    for row in data_str:
        print(format_row(row))

def get_year(ts):
    return datetime.fromtimestamp(int(ts)).year

ratings = sc.textFile("ratings_1.txt") \
    .union(sc.textFile("ratings_2.txt")) \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (get_year(x[3]), (float(x[2]),1)))

agg = ratings.reduceByKey(lambda a,b:(a[0]+b[0], a[1]+b[1]))

result = agg.map(lambda x: (x[0], round(x[1][0]/x[1][1],2), x[1][1])) \
            .sortBy(lambda x: x[0])

top = result.take(20)
print_table(top, ["Year","AvgRating","Count"])

output = result.map(lambda x: f"{x[0]},{x[1]},{x[2]}")
output.coalesce(1).saveAsTextFile("assignment6/output")

EOF

In [13]:
!rm -rf assignment6/output
!spark-submit assignment6/Assignment6.py ratings_1.txt ratings_2.txt

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 11:52:56 INFO SparkContext: Running Spark version 4.0.2
26/04/22 11:52:56 INFO SparkContext: OS info Linux, 6.6.113+, amd64
26/04/22 11:52:56 INFO SparkContext: Java version 17.0.18
26/04/22 11:52:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/22 11:52:57 INFO ResourceUtils: ==============================================================
26/04/22 11:52:57 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/22 11:52:57 INFO ResourceUtils: ==============================================================
26/04/22 11:52:57 INFO SparkContext: Submitted application: Assignment6
26/04/22 11:52:57 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: